In [45]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Load the dataset
df = pd.read_csv("../data/raw/housing.csv")
print(df.head())

# Stratified test set
df["income_cat"] = pd.cut(
    df["median_income"], bins=[0, 1.5, 3.0, 4.5, 6.0, np.inf], labels=[1, 2, 3, 4, 5]
)

split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_idx, test_idx in split.split(df, df["income_cat"]):
    strat_train_set = df.loc[train_idx].drop("income_cat", axis=1)
    strat_test_set = df.loc[test_idx].drop("income_cat", axis=1)

housing = strat_train_set.copy()

# Separate features and labels

housing_labels = housing["median_house_value"].copy()
housing = housing.drop("median_house_value", axis=1)

# Separate numerical and categorical columns
housing_num = housing.select_dtypes(
    include="number"
).columns.tolist()
housing_cat = ["ocean_proximity"]

# Pipeline for numerical cols

num_pipeline = Pipeline(
    [("impute", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
)

# Pipeline for categorical columns

cat_pipeline = Pipeline([("onehot", OneHotEncoder(handle_unknown="ignore"))])

# Construct the full pipeline

full_pipeline = ColumnTransformer(
    [("num", num_pipeline, housing_num), ("cat", cat_pipeline, housing_cat)]
)

# Transform the data

housing_processed = full_pipeline.fit_transform(housing)

   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0    -122.23     37.88                41.0        880.0           129.0   
1    -122.22     37.86                21.0       7099.0          1106.0   
2    -122.24     37.85                52.0       1467.0           190.0   
3    -122.25     37.85                52.0       1274.0           235.0   
4    -122.25     37.85                52.0       1627.0           280.0   

   population  households  median_income  median_house_value ocean_proximity  
0       322.0       126.0         8.3252            452600.0        NEAR BAY  
1      2401.0      1138.0         8.3014            358500.0        NEAR BAY  
2       496.0       177.0         7.2574            352100.0        NEAR BAY  
3       558.0       219.0         5.6431            341300.0        NEAR BAY  
4       565.0       259.0         3.8462            342200.0        NEAR BAY  


## Model Training

In [46]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

# Linear Regression model
lin_reg = LinearRegression()
lin_reg.fit(housing_processed, housing_labels)
lin_preds = lin_reg.predict(housing_processed)
lin_rmse = root_mean_squared_error(housing_labels, lin_preds)
print("RMSE of Linear Regressor is:- ", lin_rmse)

# Decision Tree Regression model
dt_reg = DecisionTreeRegressor()
dt_reg.fit(housing_processed, housing_labels)
dt_preds = dt_reg.predict(housing_processed)
dt_rmse = root_mean_squared_error(housing_labels, dt_preds)
print("RMSE of Decision Tree Regressor is:- ", dt_rmse)

# Random Forest Regression model
rf_reg = RandomForestRegressor()
rf_reg.fit(housing_processed, housing_labels)
rf_preds = rf_reg.predict(housing_processed)
rf_rmse = root_mean_squared_error(housing_labels, rf_preds)
print("RMSE of Random Forest Regressor is:- ", rf_rmse)

RMSE of Linear Regressor is:-  69050.56219504567
RMSE of Decision Tree Regressor is:-  0.0
RMSE of Random Forest Regressor is:-  18419.49916533284


## Cross Validation

In [47]:
from sklearn.model_selection import cross_val_score

# Linear Regression
lin_scores = cross_val_score(
    lin_reg,
    housing_processed,
    housing_labels,
    scoring="neg_root_mean_squared_error",
    cv=5
)
lin_rmse_scores = -lin_scores

print("Linear Regression:")
print("Scores:", lin_rmse_scores)
print("Mean:", lin_rmse_scores.mean())
print("Standard deviation:", lin_rmse_scores.std())

# Decision Tree
dt_scores = cross_val_score(
    dt_reg,
    housing_processed,
    housing_labels,
    scoring="neg_root_mean_squared_error",
    cv=5
)
dt_rmse_scores = -dt_scores

print("\nDecision Tree:")
print(dt_rmse_scores)
print(dt_rmse_scores.mean())
print(dt_rmse_scores.std())

# Random Forest
rf_scores = cross_val_score(
    rf_reg,
    housing_processed,
    housing_labels,
    scoring="neg_root_mean_squared_error",
    cv=5
)
rf_rmse_scores = -rf_scores

print("\nRandom Forest:")
print(rf_rmse_scores)
print(rf_rmse_scores.mean())
print(rf_rmse_scores.std())

Linear Regression:
Scores: [68906.16750279 68244.5107051  69961.21623633 70052.1498968
 68928.22700028]
Mean: 69218.4542682591
Standard deviation: 689.5019503176177

Decision Tree:
[69006.40597654 67626.51084524 70754.85234433 70941.694543
 69635.74592926]
69593.04192767544
1215.0619070788325

Random Forest:
[50102.09971236 48776.81568807 49003.0647989  50817.42976583
 50726.50372922]
49885.18273887744
852.1230052962761
